# 03 — Motor de benchmarking: C4.5 y Random Forest

Esta HU construye el motor para ejecutar grillas de parámetros. Reutiliza las
definiciones de los notebooks 01 y 02 sin mantener copias de los modelos. Las
baterías experimentales, los análisis y las gráficas corresponden a HU-11/12/13.

Ejecutar desde la raíz del repositorio o desde `notebooks/`, con el repositorio
completo disponible en Colab. Las entradas son particiones ya preparadas y una
grilla independiente por modelo. El motor no selecciona el mejor modelo.


In [ ]:
from pathlib import Path
from time import perf_counter_ns
from types import ModuleType
import json
import sys

import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import ParameterGrid, train_test_split

for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos_tp.py").is_file() and (carpeta / "notebooks/01_decision_tree_c45.ipynb").is_file():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError("Ejecutá desde la raíz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from datos_tp import SEMILLA, cargar_datasets, particionar
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.validation import check_is_fitted


def cargar_definiciones(ruta, etiqueta, nombre_modulo):
    """Ejecuta solo celdas de definiciones del notebook original, en orden."""
    modulo = ModuleType(nombre_modulo)
    sys.modules[nombre_modulo] = modulo  # dataclass necesita encontrar el módulo.
    celdas = json.loads(ruta.read_text(encoding="utf-8"))["cells"]
    seleccionadas = [c for c in celdas if etiqueta in c.get("metadata", {}).get("tags", [])]
    if not seleccionadas:
        raise ValueError(f"No hay celdas {etiqueta} en {ruta.name}.")
    if etiqueta == "rf-definiciones":
        modulo.__dict__.update({
            "pd": pd, "ColumnTransformer": ColumnTransformer,
            "RandomForestClassifier": RandomForestClassifier, "Pipeline": Pipeline,
            "OneHotEncoder": OneHotEncoder, "check_is_fitted": check_is_fitted,
            "SEMILLA": SEMILLA,
        })
    for celda in seleccionadas:
        exec(compile("".join(celda["source"]), str(ruta), "exec"), modulo.__dict__)
    return modulo


c45 = cargar_definiciones(RAIZ / "notebooks/01_decision_tree_c45.ipynb",
                         "c45-definiciones", "c45_para_benchmark")
rf = cargar_definiciones(RAIZ / "notebooks/02_random_forest.ipynb",
                        "rf-definiciones", "rf_para_benchmark")


## Contrato experimental

`ejecutar_benchmark` recibe un nombre de dataset, `X` e `y` de entrenamiento y
evaluación, dos grillas y una semilla. `y` conserva el formato de HU-04: un
DataFrame con columna `objetivo`; `predict(X)` devuelve `prediccion` con el mismo
índice. Los conjuntos deben ser disjuntos y tener las mismas características.
El llamador indica la semilla y el protocolo usados para crear las particiones.

Cada fila del resultado representa una configuración y un conjunto evaluado.
`parametros` conserva los valores de la grilla y `parametros_efectivos` incluye
también los valores predeterminados. `matriz_confusion` es una lista de listas;
`clases` fija el orden de sus filas y columnas. Precision, Recall y F1 usan
promedios macro y weighted con `zero_division=0`. Accuracy es una sola cifra.

El tiempo de entrenamiento abarca `.fit`, incluido el preprocesamiento de RF. El
de inferencia abarca `predict(X)` para todos los casos del conjunto indicado e
incluye el preprocesamiento de RF. Excluyen métricas y tablas. Se usa un solo hilo
en RF (`n_jobs=1`). Los tiempos dependen del equipo y pueden variar entre corridas.

La selección de configuraciones debe hacerse en validación dentro de entrenamiento.
La partición test queda reservada para evaluar una configuración elegida; el motor
no hace esa selección por sí mismo. Para RF, entradas con nulos requieren un
tratamiento previo explícito, ajustado únicamente con entrenamiento.


In [ ]:
PARAMETROS = {
    "C4.5": {"threshold", "max_depth", "min_samples_split", "min_gain_ratio"},
    "Random Forest": {"n_estimators", "max_features", "max_depth", "criterion", "bootstrap"},
}


def _validar_conjunto(X, y, nombre):
    if not isinstance(X, pd.DataFrame) or not isinstance(y, pd.DataFrame):
        raise TypeError(f"{nombre}: X e y deben ser DataFrames.")
    if X.empty or y.shape != (len(X), 1) or list(y.columns) != ["objetivo"]:
        raise ValueError(f"{nombre}: X debe ser no vacío e y debe tener columna objetivo.")
    if not X.index.is_unique or not X.index.equals(y.index):
        raise ValueError(f"{nombre}: los índices de X e y deben ser únicos e iguales.")
    if y["objetivo"].isna().any():
        raise ValueError(f"{nombre}: y contiene etiquetas faltantes.")


def _configuraciones(grillas):
    if not isinstance(grillas, dict) or set(grillas) != set(PARAMETROS):
        raise ValueError("grillas debe contener C4.5 y Random Forest.")
    for modelo, grilla in grillas.items():
        if not isinstance(grilla, dict):
            raise TypeError(f"{modelo}: la grilla debe ser un diccionario.")
        desconocidos = set(grilla) - PARAMETROS[modelo]
        if desconocidos:
            raise ValueError(f"{modelo}: parámetros no admitidos: {sorted(desconocidos)}.")
        for parametro, valores in grilla.items():
            if isinstance(valores, (str, bytes)) or not isinstance(valores, (list, tuple)) or not valores:
                raise ValueError(f"{modelo}.{parametro}: se requiere una lista no vacía de valores.")
        for config in ParameterGrid(grilla):
            yield modelo, config


def ejecutar_benchmark(nombre_dataset, X_train, y_train, X_eval, y_eval, grillas,
                       *, nombre_evaluacion="validacion", semilla=SEMILLA,
                       semilla_particion=None, protocolo_particion="particiones provistas"):
    """Una fila por configuración y conjunto; reutiliza ambos modelos originales."""
    if not isinstance(nombre_dataset, str) or not nombre_dataset:
        raise ValueError("nombre_dataset debe ser un texto no vacío.")
    if nombre_evaluacion not in {"validacion", "test"}:
        raise ValueError("nombre_evaluacion debe ser validacion o test.")
    if not isinstance(semilla, int) or isinstance(semilla, bool) or semilla < 0:
        raise ValueError("semilla debe ser un entero no negativo.")
    if semilla_particion is not None and (not isinstance(semilla_particion, int)
                                          or isinstance(semilla_particion, bool) or semilla_particion < 0):
        raise ValueError("semilla_particion debe ser None o un entero no negativo.")
    if not isinstance(protocolo_particion, str) or not protocolo_particion.strip():
        raise ValueError("protocolo_particion debe describir la partición.")
    _validar_conjunto(X_train, y_train, "entrenamiento")
    _validar_conjunto(X_eval, y_eval, nombre_evaluacion)
    if not X_train.columns.equals(X_eval.columns):
        raise ValueError("Las columnas de entrenamiento y evaluación deben coincidir en orden.")
    if not X_train.index.intersection(X_eval.index).empty:
        raise ValueError("Entrenamiento y evaluación deben tener índices disjuntos.")
    configuraciones = list(_configuraciones(grillas))
    clases = sorted(set(y_train["objetivo"]) | set(y_eval["objetivo"]))
    filas = []
    for numero, (nombre_modelo, config) in enumerate(configuraciones, start=1):
        try:
            if nombre_modelo == "C4.5":
                modelo = c45.ArbolC45(**config)
                parametros_efectivos = {
                    "threshold": modelo.threshold, "max_depth": modelo.max_depth,
                    "min_samples_split": modelo.min_samples_split,
                    "min_gain_ratio": modelo.min_gain_ratio,
                }
                entrenar = lambda: modelo.fit(X_train, y_train)
                predecir = modelo.predict
            else:
                modelo = rf.crear_modelo(X_train, **config, random_state=semilla)
                parametros_efectivos = modelo.named_steps["random_forest"].get_params()
                parametros_efectivos = {k: parametros_efectivos[k] for k in
                    ("n_estimators", "max_features", "max_depth", "criterion", "bootstrap", "random_state")}
                entrenar = lambda: modelo.fit(X_train, y_train["objetivo"])
            inicio = perf_counter_ns()
            entrenar()
            tiempo_entrenamiento_ms = (perf_counter_ns() - inicio) / 1_000_000
            if nombre_modelo == "Random Forest":
                predecir = rf.crear_predictor(modelo)
            for conjunto, X, y in (("entrenamiento", X_train, y_train),
                                    (nombre_evaluacion, X_eval, y_eval)):
                inicio = perf_counter_ns()
                pred = predecir(X)
                tiempo_inferencia_ms = (perf_counter_ns() - inicio) / 1_000_000
                if not isinstance(pred, pd.DataFrame) or list(pred.columns) != ["prediccion"] or not pred.index.equals(X.index):
                    raise ValueError("predict(X) no respetó el contrato compartido.")
                real = y["objetivo"]
                estimado = pred["prediccion"]
                p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
                    real, estimado, labels=clases, average="macro", zero_division=0)
                p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
                    real, estimado, labels=clases, average="weighted", zero_division=0)
                filas.append({
                    "ejecucion": numero, "dataset": nombre_dataset,
                    "modelo": nombre_modelo, "parametros": config.copy(),
                    "parametros_efectivos": parametros_efectivos.copy(),
                    "semilla": semilla, "semilla_particion": semilla_particion,
                    "protocolo_particion": protocolo_particion,
                    "conjunto": conjunto, "n_casos": len(X),
                    "accuracy": accuracy_score(real, estimado),
                    "precision_macro": p_macro, "recall_macro": r_macro, "f1_macro": f_macro,
                    "precision_weighted": p_weighted, "recall_weighted": r_weighted, "f1_weighted": f_weighted,
                    "clases": tuple(clases),
                    "matriz_confusion": confusion_matrix(real, estimado, labels=clases).tolist(),
                    "tiempo_entrenamiento_ms": tiempo_entrenamiento_ms,
                    "tiempo_inferencia_ms": tiempo_inferencia_ms,
                })
        except (TypeError, ValueError) as error:
            raise ValueError(f"{nombre_modelo}, configuración {config}: {error}") from error
    return pd.DataFrame(filas)


## Comprobación mínima

Se usa Iris y se crea validación estratificada solo desde el entrenamiento de
HU-04. El test original no interviene. Dos configuraciones por modelo producen
ocho filas: entrenamiento y validación por cada configuración. Esta ejecución
comprueba el motor; la exploración completa y la selección quedan en HU-11/13.


In [ ]:
datos = cargar_datasets()["iris"]
partes = particionar(datos["X"], datos["y"])
X_train, X_valid, y_train, y_valid = train_test_split(
    partes["X_train"], partes["y_train"], test_size=0.25,
    random_state=SEMILLA, stratify=partes["y_train"]["objetivo"])
grillas_minimas = {
    "C4.5": {"max_depth": [1, 3]},
    "Random Forest": {"n_estimators": [10, 20]},
}
resultados = ejecutar_benchmark("iris", X_train, y_train, X_valid, y_valid,
                                grillas_minimas, semilla=SEMILLA, semilla_particion=SEMILLA,
                                protocolo_particion="HU-04 80/20 estratificado; validación 25% del entrenamiento")
assert len(resultados) == 8
assert set(resultados["conjunto"]) == {"entrenamiento", "validacion"}
assert resultados.groupby("modelo")["ejecucion"].nunique().to_dict() == {
    "C4.5": 2, "Random Forest": 2}
display(resultados.drop(columns=["parametros_efectivos", "matriz_confusion"]))
